# Essential EDA Methods — Teaching Notebook

This notebook uses a **customer behavior dataset** created specifically for teaching **Exploratory Data Analysis (EDA)**.

## Learning goals
By the end of this notebook, students should be able to:
- inspect a dataset quickly
- understand shape, columns, and index
- inspect data types and memory usage
- compute summary statistics
- profile missing values
- detect duplicates
- summarize categorical columns
- compute correlations
- perform grouped aggregations
- scan for outliers using **IQR** and **z-score**
- apply simple type-fixing and cleaning steps

> Dataset file used: `customer_behavior_eda.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make notebook output easier to read
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 0. Load the dataset

In [ ]:
df = pd.read_csv('customer_behavior_eda.csv')
df.head()

## 1. Quick Peek at the Data

We usually start with:
- `head()` → first rows
- `tail()` → last rows
- `sample()` → random rows

This helps us check whether loading worked correctly and whether the values look reasonable.


In [ ]:
print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nRandom sample:")
display(df.sample(5, random_state=42))


## 2. Shape and Index

This tells us:
- how many rows and columns we have
- what the row index looks like
- what the column names are


In [ ]:
print("Shape:", df.shape)
print("\nIndex:")
display(df.index)

print("\nColumns:")
display(df.columns.tolist())


## 3. Types and Basic Info

This step helps reveal:
- numbers stored as strings
- missing values
- memory usage
- mixed or unexpected types


In [ ]:
print("Data types:")
display(df.dtypes)

print("\nInfo:")
df.info()

print("\nTotal memory usage (bytes):", df.memory_usage(deep=True).sum())


### Observation
Notice that some columns that look numeric may actually be stored as **object/string** type.
For example, `annual_income_egp` may contain commas like `"216,154"` and therefore is not numeric yet.


## 4. Summary Statistics (Numeric)

This gives a quick numerical summary:
- count
- mean
- std
- min / max
- percentiles

Useful for spotting skewed data and possible outliers.


In [ ]:
df.describe()


In [ ]:
df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])


## 5. Missing-Values Profile

We check:
- count of missing values
- percentage of missing values

This helps us decide:
- drop?
- fill?
- investigate?


In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_ratio = df.isna().mean().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_ratio': (missing_ratio * 100).round(2).astype(str) + '%'
})

missing_summary


## 6. Duplicates Check

Duplicates can distort:
- counts
- averages
- model performance
- train/test splits

We check total duplicated rows first.


In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)


In [ ]:
# Keep a cleaned copy without duplicated rows
df_nodup = df.drop_duplicates().copy()
print("Shape before removing duplicates:", df.shape)
print("Shape after removing duplicates:", df_nodup.shape)


## 7. Cardinality and Categorical Summaries

This helps detect:
- ID-like columns
- low-cardinality categories
- class imbalance
- messy labels (`" cairo "`, `"GIZA"`, etc.)


In [ ]:
print("Distinct values per column:")
display(df_nodup.nunique())

print("\nValue counts for region:")
display(df_nodup['region'].value_counts(dropna=False))

print("\nValue counts for acquisition_channel:")
display(df_nodup['acquisition_channel'].value_counts(dropna=False))

print("\nValue counts for churned:")
display(df_nodup['churned'].value_counts(dropna=False))


## 8. Basic Type Fixes and Categorical Hygiene

Before deeper EDA, we often fix:
- strings that should be numeric
- strings that should be datetime
- extra spaces / inconsistent capitalization in categories


In [ ]:
df_clean = df_nodup.copy()

# Fix annual_income_egp from strings like "216,154" to numeric
df_clean['annual_income_egp'] = (
    df_clean['annual_income_egp']
    .astype(str)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
)
df_clean['annual_income_egp'] = pd.to_numeric(df_clean['annual_income_egp'], errors='coerce')

# Convert signup_date to datetime
df_clean['signup_date'] = pd.to_datetime(df_clean['signup_date'], errors='coerce')

# Clean text columns
df_clean['region'] = df_clean['region'].astype(str).str.strip().str.title().replace('Nan', np.nan)
df_clean['acquisition_channel'] = df_clean['acquisition_channel'].astype(str).str.strip().str.title()

print(df_clean.dtypes)
display(df_clean.head())


### Datetime feature examples

In [ ]:
df_clean['signup_year'] = df_clean['signup_date'].dt.year
df_clean['signup_month'] = df_clean['signup_date'].dt.month
df_clean['signup_dayofweek'] = df_clean['signup_date'].dt.day_name()

df_clean[['signup_date', 'signup_year', 'signup_month', 'signup_dayofweek']].head()


## 9. Correlations (Numeric ↔ Numeric)

Correlation helps us detect:
- variables moving together
- redundancy
- possible predictive signal
- multicollinearity risk


In [ ]:
corr = df_clean.corr(numeric_only=True, method='pearson')
corr


In [ ]:
# Simple heatmap using matplotlib only
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(corr, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Correlation Matrix")
plt.colorbar(im)
plt.tight_layout()
plt.show()


## 10. Grouped Aggregations

This is one of the most important EDA tools.

We can ask questions like:
- Which region has the highest average monthly spend?
- Which segment has the highest churn rate?
- Which acquisition channel brings more active customers?


In [ ]:
region_summary = (
    df_clean.groupby('region')
    .agg({
        'customer_id': 'count',
        'monthly_spend_egp': ['mean', 'median'],
        'support_tickets': 'mean',
        'satisfaction_score': 'mean'
    })
    .sort_values(('monthly_spend_egp', 'mean'), ascending=False)
)

region_summary


In [ ]:
# Churn rate by segment
segment_churn = (
    df_clean.assign(churned_flag=(df_clean['churned'] == 'Yes').astype(int))
    .groupby('segment')
    .agg(
        customers=('customer_id', 'count'),
        avg_monthly_spend=('monthly_spend_egp', 'mean'),
        churn_rate=('churned_flag', 'mean')
    )
    .sort_values('churn_rate', ascending=False)
)

segment_churn['churn_rate'] = (segment_churn['churn_rate'] * 100).round(2)
segment_churn


In [ ]:
# Example: grouped by acquisition channel
channel_summary = (
    df_clean.groupby('acquisition_channel')
    .agg({
        'customer_id': 'count',
        'website_visits_last_30d': 'mean',
        'monthly_spend_egp': 'mean',
        'orders_last_6m': 'mean'
    })
    .sort_values(('monthly_spend_egp'), ascending=False)
)

channel_summary


## 11. Outlier Scanning

Two common approaches:
- **IQR rule**
- **z-score rule**


### 11.1 IQR method on `monthly_spend_egp`

In [ ]:
s = df_clean['monthly_spend_egp'].dropna()

q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1 =", q1)
print("Q3 =", q3)
print("IQR =", iqr)
print("Lower bound =", lower)
print("Upper bound =", upper)

outliers_iqr = df_clean[(df_clean['monthly_spend_egp'] < lower) | (df_clean['monthly_spend_egp'] > upper)]
print("Number of IQR outliers:", len(outliers_iqr))
display(outliers_iqr[['customer_id', 'monthly_spend_egp']].sort_values('monthly_spend_egp', ascending=False).head(10))


### 11.2 Z-score method on `website_visits_last_30d`

In [ ]:
s2 = df_clean['website_visits_last_30d'].dropna()
z_scores = ((s2 - s2.mean()) / s2.std()).abs()

outlier_mask = z_scores > 3
outliers_z = df_clean.loc[s2.index[outlier_mask], ['customer_id', 'website_visits_last_30d']]

print("Number of z-score outliers:", len(outliers_z))
display(outliers_z.sort_values('website_visits_last_30d', ascending=False))


## 12. Optional: Very Simple Visual Checks

Even though this notebook focuses mainly on tabular EDA, quick plots can help.


In [ ]:
df_clean['monthly_spend_egp'].hist(bins=30, figsize=(7,4))
plt.title('Distribution of Monthly Spend')
plt.xlabel('monthly_spend_egp')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
df_clean['region'].value_counts(dropna=False).plot(kind='bar', figsize=(7,4))
plt.title('Customers by Region')
plt.xlabel('Region')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 13. Minimal EDA Checklist Recap

1. Peek at the data  
2. Check shape, index, columns  
3. Inspect data types and memory  
4. Run summary statistics  
5. Profile missing values  
6. Check duplicates  
7. Examine categorical cardinality and frequencies  
8. Compute correlations  
9. Run groupby aggregations  
10. Scan for outliers  
